# 04 — Difference-in-Differences Analysis

This notebook implements the core causal inference methods:
1. Classic 2×2 DiD (pre/post × expansion/non-expansion)
2. Two-Way Fixed Effects (TWFE) with staggered treatment
3. Event Study estimation and visualization
4. Discussion of TWFE bias and motivation for modern estimators (see R scripts)

**Input:** `data/processed/analysis_panel.csv`  
**Output:** Regression tables, event study plots

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS
import warnings
import os

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 150,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

os.makedirs('../figures', exist_ok=True)
os.makedirs('../tables', exist_ok=True)

# Load data
panel = pd.read_csv('../data/processed/analysis_panel.csv')
print(f"Panel: {panel.shape}")
print(f"States: {panel['state'].nunique()}, Years: {panel['year'].min()}–{panel['year'].max()}")

In [ ]:
# Identify available outcome variables
outcome_candidates = [
    'allcause_age_adj_rate', 'allcause_crude_rate',
    'diabetes_age_adj_rate', 'diabetes_crude_rate', 'diabetes_prevalence',
    'maternal_age_adj_rate', 'maternal_crude_rate',
    'low_birth_weight_pct'
]
outcome_vars = [c for c in outcome_candidates if c in panel.columns and panel[c].notna().sum() > 100]

# Identify available control variables
control_candidates = [
    'median_household_income', 'poverty_rate', 'total_population',
    'pct_white', 'pct_black', 'pct_hispanic',
    'uninsured_rate', 'unemployment_rate'
]
control_vars = [c for c in control_candidates if c in panel.columns and panel[c].notna().sum() > 100]

print(f"Outcome variables available: {outcome_vars if outcome_vars else 'NONE — download CDC data first'}")
print(f"Control variables available: {control_vars if control_vars else 'NONE — get Census API key first'}")

if not outcome_vars:
    print("\n⚠️  No outcome data found. This notebook needs CDC data to run the regressions.")
    print("For now, we'll demonstrate the methodology using control variables as placeholder outcomes.")
    # Use ACS vars as demo outcomes if no health outcomes available
    outcome_vars = [c for c in ['poverty_rate', 'median_household_income'] if c in panel.columns]

---
## 1. Classic 2×2 Difference-in-Differences

The simplest DiD: compare changes in outcomes before/after 2014 for expansion vs non-expansion states.

$$Y_{st} = \alpha + \beta_1 \cdot Expansion_s + \beta_2 \cdot Post_t + \delta \cdot (Expansion_s \times Post_t) + \epsilon_{st}$$

- $\delta$ is the DiD estimate (our causal parameter of interest)

In [ ]:
def run_simple_did(panel, outcome_var, controls=None):
    """
    Run a simple 2x2 DiD regression.
    Only uses early adopters (2014) vs never-expanded states.
    """
    # Filter to clean 2x2 comparison
    # Treatment: states that expanded in 2014
    # Control: states that never expanded
    df = panel[
        (panel['expansion_year'].isin([2014, 0])) &
        (panel[outcome_var].notna())
    ].copy()
    
    df['post'] = (df['year'] >= 2014).astype(int)
    df['treat'] = (df['expansion_year'] == 2014).astype(int)
    df['treat_x_post'] = df['treat'] * df['post']
    
    # Build formula
    formula = f'{outcome_var} ~ treat + post + treat_x_post'
    if controls:
        available_controls = [c for c in controls if c in df.columns and df[c].notna().sum() > 50]
        if available_controls:
            formula += ' + ' + ' + '.join(available_controls)
    
    # Fit with clustered standard errors
    model = smf.ols(formula, data=df).fit(
        cov_type='cluster',
        cov_kwds={'groups': df['state_fips']}
    )
    
    print(f"\n{'='*60}")
    print(f"Simple DiD: {outcome_var}")
    print(f"{'='*60}")
    print(f"Treatment: 2014 expansion states | Control: never-expanded")
    print(f"N = {model.nobs:.0f} | R² = {model.rsquared:.4f}")
    print(f"\n*** DiD Estimate (treat_x_post): {model.params['treat_x_post']:.4f} ***")
    print(f"    Std Error: {model.bse['treat_x_post']:.4f}")
    print(f"    95% CI: [{model.conf_int().loc['treat_x_post', 0]:.4f}, {model.conf_int().loc['treat_x_post', 1]:.4f}]")
    print(f"    p-value: {model.pvalues['treat_x_post']:.4f}")
    
    return model

In [ ]:
# Run simple DiD for each outcome
simple_did_results = {}

for outcome in outcome_vars:
    try:
        model = run_simple_did(panel, outcome, controls=control_vars)
        simple_did_results[outcome] = model
    except Exception as e:
        print(f"\n⚠️  Error with {outcome}: {e}")

---
## 2. Two-Way Fixed Effects (TWFE) DiD

Accounts for staggered treatment timing using state and year fixed effects:

$$Y_{st} = \alpha_s + \lambda_t + \delta \cdot D_{st} + X_{st}\gamma + \epsilon_{st}$$

Where $\alpha_s$ = state fixed effects, $\lambda_t$ = year fixed effects, $D_{st}$ = treatment indicator.

In [ ]:
def run_twfe(panel, outcome_var, controls=None):
    """
    Run TWFE DiD with state and year fixed effects.
    Uses all states including staggered adopters.
    """
    df = panel[panel[outcome_var].notna()].copy()
    
    # Set panel index for linearmodels
    df = df.set_index(['state_fips', 'year'])
    
    # Build dependent and independent variables
    y = df[outcome_var]
    
    exog_vars = ['post_expansion']
    if controls:
        available = [c for c in controls if c in df.columns and df[c].notna().sum() > 50]
        exog_vars += available
    
    X = df[exog_vars].copy()
    
    # Drop rows with any NaN in X
    mask = X.notna().all(axis=1) & y.notna()
    y = y[mask]
    X = X[mask]
    
    # Fit TWFE with entity and time fixed effects
    model = PanelOLS(
        y, X,
        entity_effects=True,
        time_effects=True
    ).fit(cov_type='clustered', cluster_entity=True)
    
    print(f"\n{'='*60}")
    print(f"TWFE DiD: {outcome_var}")
    print(f"{'='*60}")
    print(f"All states (staggered adoption) | Entity + Time FE")
    print(f"N = {model.nobs} | R² (within) = {model.rsquared_within:.4f}")
    print(f"\n*** TWFE DiD Estimate (post_expansion): {model.params['post_expansion']:.4f} ***")
    print(f"    Std Error: {model.std_errors['post_expansion']:.4f}")
    print(f"    95% CI: [{model.conf_int().loc['post_expansion', 'lower']:.4f}, {model.conf_int().loc['post_expansion', 'upper']:.4f}]")
    print(f"    p-value: {model.pvalues['post_expansion']:.4f}")
    
    return model

In [ ]:
# Run TWFE for each outcome
twfe_results = {}

for outcome in outcome_vars:
    try:
        model = run_twfe(panel, outcome, controls=control_vars)
        twfe_results[outcome] = model
    except Exception as e:
        print(f"\n⚠️  Error with {outcome}: {e}")

---
## 3. Event Study

Estimate dynamic treatment effects by year relative to expansion:

$$Y_{st} = \alpha_s + \lambda_t + \sum_{k \neq -1} \delta_k \cdot \mathbb{1}[t - E_s = k] + X_{st}\gamma + \epsilon_{st}$$

This is the most informative visualization — it shows:
- **Pre-treatment coefficients** → test parallel trends (should be ~0)
- **Post-treatment coefficients** → dynamic causal effects

In [ ]:
def run_event_study(panel, outcome_var, controls=None, leads=4, lags=8):
    """
    Estimate event study with leads/lags relative to treatment.
    Omits t=-1 as the reference period.
    """
    # Filter to states with expansion info
    df = panel[
        (panel[outcome_var].notna()) &
        (panel['event_time'].notna() | (panel['ever_expanded'] == 0))
    ].copy()
    
    # For never-treated states, they serve as controls
    # Create event time dummies only for treated states
    # Bin endpoints
    df['event_time_binned'] = df['event_time'].copy()
    df.loc[df['event_time_binned'] < -leads, 'event_time_binned'] = -leads
    df.loc[df['event_time_binned'] > lags, 'event_time_binned'] = lags
    
    # Create dummies (excluding t=-1 as reference)
    event_times = sorted([t for t in df['event_time_binned'].dropna().unique() if t != -1])
    
    for t in event_times:
        col_name = f'et_{int(t)}' if t < 0 else f'et_plus_{int(t)}'
        df[col_name] = (df['event_time_binned'] == t).astype(int)
    
    # Set up panel
    df_panel = df.set_index(['state_fips', 'year'])
    y = df_panel[outcome_var]
    
    # Event time dummies
    et_cols = [c for c in df_panel.columns if c.startswith('et_')]
    exog_vars = et_cols.copy()
    
    if controls:
        available = [c for c in controls if c in df_panel.columns and df_panel[c].notna().sum() > 50]
        exog_vars += available
    
    X = df_panel[exog_vars].copy()
    mask = X.notna().all(axis=1) & y.notna()
    y = y[mask]
    X = X[mask]
    
    # Fit
    model = PanelOLS(
        y, X,
        entity_effects=True,
        time_effects=True
    ).fit(cov_type='clustered', cluster_entity=True)
    
    # Extract event study coefficients
    es_coefs = []
    for t in event_times:
        col_name = f'et_{int(t)}' if t < 0 else f'et_plus_{int(t)}'
        if col_name in model.params.index:
            es_coefs.append({
                'event_time': t,
                'coef': model.params[col_name],
                'se': model.std_errors[col_name],
                'ci_lower': model.conf_int().loc[col_name, 'lower'],
                'ci_upper': model.conf_int().loc[col_name, 'upper'],
                'pvalue': model.pvalues[col_name]
            })
    
    # Add reference period (t=-1)
    es_coefs.append({
        'event_time': -1, 'coef': 0, 'se': 0,
        'ci_lower': 0, 'ci_upper': 0, 'pvalue': np.nan
    })
    
    es_df = pd.DataFrame(es_coefs).sort_values('event_time')
    
    return model, es_df

In [ ]:
def plot_event_study(es_df, outcome_var, title=None):
    """
    Create a publication-quality event study plot.
    """
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Confidence intervals
    ax.fill_between(es_df['event_time'], es_df['ci_lower'], es_df['ci_upper'],
                    alpha=0.2, color='#2171b5')
    
    # Point estimates
    ax.plot(es_df['event_time'], es_df['coef'], 'o-',
            color='#2171b5', linewidth=2, markersize=6)
    
    # Reference lines
    ax.axhline(y=0, color='black', linewidth=0.8)
    ax.axvline(x=-0.5, color='red', linestyle='--', alpha=0.6, linewidth=1.5)
    
    # Annotate reference period
    ax.annotate('Reference\nperiod (t=-1)',
                xy=(-1, 0), xytext=(-2.5, ax.get_ylim()[1] * 0.7),
                arrowprops=dict(arrowstyle='->', color='gray'),
                fontsize=9, color='gray', ha='center')
    
    # Labels
    ax.text(-2.5, ax.get_ylim()[0] * 0.9, 'Pre-treatment',
            ha='center', fontsize=10, color='gray', style='italic')
    ax.text(3, ax.get_ylim()[0] * 0.9, 'Post-treatment',
            ha='center', fontsize=10, color='gray', style='italic')
    
    if title is None:
        title = outcome_var.replace('_', ' ').title()
    ax.set_title(f'Event Study: {title}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Years Relative to Medicaid Expansion', fontsize=12)
    ax.set_ylabel('Estimated Effect (95% CI)', fontsize=12)
    
    plt.tight_layout()
    return fig

In [ ]:
# Run event studies for all available outcomes
event_study_results = {}

for outcome in outcome_vars:
    try:
        model, es_df = run_event_study(panel, outcome, controls=control_vars)
        event_study_results[outcome] = (model, es_df)
        
        # Print results
        print(f"\n{'='*60}")
        print(f"Event Study: {outcome}")
        print(f"{'='*60}")
        print(es_df[['event_time', 'coef', 'se', 'ci_lower', 'ci_upper', 'pvalue']].to_string(index=False))
        
        # Plot
        fig = plot_event_study(es_df, outcome)
        fig.savefig(f'../figures/event_study_{outcome}.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Saved: figures/event_study_{outcome}.png")
        
    except Exception as e:
        print(f"\n⚠️  Error with {outcome}: {e}")

---
## 4. Results Summary Table

In [ ]:
# Compile results into a summary table
results_rows = []

for outcome in outcome_vars:
    row = {'outcome': outcome}
    
    # Simple DiD
    if outcome in simple_did_results:
        m = simple_did_results[outcome]
        row['simple_did_coef'] = m.params.get('treat_x_post', np.nan)
        row['simple_did_se'] = m.bse.get('treat_x_post', np.nan)
        row['simple_did_pval'] = m.pvalues.get('treat_x_post', np.nan)
    
    # TWFE
    if outcome in twfe_results:
        m = twfe_results[outcome]
        row['twfe_coef'] = m.params.get('post_expansion', np.nan)
        row['twfe_se'] = m.std_errors.get('post_expansion', np.nan)
        row['twfe_pval'] = m.pvalues.get('post_expansion', np.nan)
    
    results_rows.append(row)

results_df = pd.DataFrame(results_rows)

if len(results_df) > 0:
    print("\nResults Summary")
    print("=" * 80)
    display(results_df.round(4))
    
    # Save
    results_df.to_csv('../tables/main_results.csv', index=False)
    print("\nSaved: tables/main_results.csv")

---
## 5. TWFE Bias Discussion

### Why TWFE can be biased with staggered treatment

Recent econometrics literature (Goodman-Bacon 2021, de Chaisemartin & D'Haultfœuille 2020) shows that the standard TWFE estimator with staggered adoption can produce **biased estimates** because it implicitly uses already-treated units as controls ("bad comparisons").

The bias arises when:
1. Treatment effects are **heterogeneous** across cohorts or over time
2. Treatment timing is **staggered** (which it is in our setting)

### Solutions implemented in R scripts:

1. **Callaway & Sant'Anna (2021)** — `R/01_did_callaway_santanna.R`
   - Estimates group-time ATTs avoiding bad comparisons
   - Uses `did` package

2. **Sun & Abraham (2021)** — `R/03_sun_abraham.R`
   - Interaction-weighted estimator
   - Uses `fixest` package

3. **Goodman-Bacon Decomposition** — `R/04_twfe_diagnostics.R`
   - Decomposes TWFE into sub-estimates to diagnose bias
   - Uses `bacondecomp` package

### Proceed to:
- **R scripts** in `R/` folder for modern DiD estimators
- **`05_event_study.ipynb`** (optional) for additional event study visualizations
- **`06_synthetic_control.ipynb`** for single-state deep dive
- **`07_robustness_checks.ipynb`** for sensitivity analyses